# 01 — Raw audit, deterministic cleaning and coverage
Run on Dataproc Python 3. All exclusions are explicit; original CSVs remain unchanged.

In [ ]:
from pathlib import Path
import hashlib
import json
import os
import subprocess
import sys
import zipfile

from IPython.display import display

BUCKET = os.environ.get("BUCKET", "sg-rainfall-dsa5208-261011-data")
def cloud_read(args):
    result = subprocess.run(["gcloud", *args], capture_output=True, text=True)
    if result.returncode:
        raise RuntimeError("Cloud read failed; check the authorised project, object and VM permissions.")
    return result.stdout
EXPECTED_RUN_ID = "20261012-full-01"
runtime_uri = f"gs://{BUCKET}/runs/{EXPECTED_RUN_ID}/config/runtime.json"
runtime = json.loads(cloud_read(["storage", "cat", runtime_uri]))
if runtime.get("run_id") != EXPECTED_RUN_ID:
    raise RuntimeError("Runtime manifest belongs to a different run.")
if runtime["billing_account_id"] != "01DAD8-AE5F1A-E3910A" or runtime["credit_id"] != "3LU63BPGY77JFDQQ":
    raise RuntimeError("Unexpected billing account or credit.")
PROJECT_ID = runtime["project_id"]
RUN_ID = runtime["run_id"]
ROOT = f"gs://{BUCKET}/runs/{RUN_ID}"
PROJECT_ROOT = Path.home() / "rainfall-project"
PROJECT_ROOT.mkdir(exist_ok=True)
bundle = PROJECT_ROOT / "source_bundle.zip"
cloud_read(["storage", "cp", runtime["source_bundle_uri"], str(bundle)])
if hashlib.sha256(bundle.read_bytes()).hexdigest() != runtime["source_bundle_sha256"]:
    raise RuntimeError("Source bundle hash differs from runtime manifest.")
with zipfile.ZipFile(bundle) as archive:
    for member in archive.namelist():
        if not (PROJECT_ROOT / member).resolve().is_relative_to(PROJECT_ROOT.resolve()):
            raise ValueError("Unsafe archive member.")
    archive.extractall(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT / "src"))
sys.path.insert(0, str(PROJECT_ROOT))
from rainfall.config import load_config, config_hash
config = load_config(PROJECT_ROOT / "config" / "experiment.json")
if config_hash(config) != runtime["config_sha256"]:
    raise RuntimeError("Experiment configuration differs from the uploaded manifest.")
billing = json.loads(cloud_read(["billing", "projects", "describe", PROJECT_ID, "--format=json"]))
from rainfall.billing import validate_runtime_credit_gate
credit_gate_status = validate_runtime_credit_gate(runtime, project_id=PROJECT_ID, billing_info=billing)
display(credit_gate_status)
from pyspark.sql import SparkSession, functions as F
from pyspark import StorageLevel
from rainfall.runtime import spark_action, environment, require_distributed
spark = (SparkSession.builder.master("yarn").appName(f"rainfall-{RUN_ID}")
    .config("spark.driver.memory", "4g")
    .config("spark.executor.memory", "4g")
    .config("spark.executor.cores", "2")
    .config("spark.executor.instances", "4")
    .config("spark.dynamicAllocation.enabled", "false")
    .config("spark.ui.retainedJobs", "10000")
    .config("spark.ui.retainedStages", "10000")
    .config("spark.sql.ui.retainedExecutions", "10000")
    .config("spark.sql.shuffle.partitions", str(config["shuffle_partitions"]))
    .getOrCreate())
spark.conf.set("spark.sql.session.timeZone", "Asia/Singapore")
spark.conf.set("spark.sql.ansi.enabled", "false")
spark.conf.set("rainfall.evidence.root", f"{ROOT}/evidence")
require_distributed(spark)
display(environment(spark))
display({"run_id": RUN_ID, "config_sha256": config_hash(config), "credit_id": runtime["credit_id"]})


In [ ]:
from scripts.run_quality_audit import read_manifest, selected_files, run_audit, make_input_frame
from google.cloud import storage

manifest_path = PROJECT_ROOT / "manifests" / "input_manifest.json"
cloud_read(["storage", "cp", f"gs://{BUCKET}/manifests/input_manifest.json", str(manifest_path)])
manifest, manifest_sha256 = read_manifest(manifest_path)
files = selected_files(manifest)
client = storage.Client(project=PROJECT_ID)
hash_results = []
for record in files:
    blob = client.bucket(BUCKET).blob(record["relative_path"])
    digest = hashlib.sha256()
    with blob.open("rb") as stream:
        for block in iter(lambda: stream.read(8 * 1024 * 1024), b""):
            digest.update(block)
    hash_results.append({"year": record["year"], "sha256_matches": digest.hexdigest() == record["sha256"]})
display(hash_results)
if not all(r["sha256_matches"] for r in hash_results):
    raise RuntimeError("At least one cloud input differs from the frozen local manifest.")

In [ ]:
with spark_action(spark, RUN_ID, "raw_quality_audit"):
    run_audit(spark, files, manifest_sha256, f"gs://{BUCKET}", f"{ROOT}/audit")
    spark.read.json(f"{ROOT}/audit/quality_by_year").orderBy("year").show(8, truncate=False)
    spark.read.json(f"{ROOT}/audit/reading_type_unit_distribution").show(30, truncate=False)

In [ ]:
from rainfall.cleaning import clean_observations, coverage
with spark_action(spark, RUN_ID, "deterministic_cleaning"):
    raw = make_input_frame(spark, files, f"gs://{BUCKET}")
    clean, rejected, quality, exclusions, alignment = clean_observations(raw, config)
    clean = clean.persist(StorageLevel.DISK_ONLY)
    clean.write.mode("errorifexists").partitionBy("year").parquet(f"{ROOT}/cleaned")
    rejected.write.mode("errorifexists").partitionBy("_source_year").parquet(f"{ROOT}/rejected")
    quality.write.mode("errorifexists").json(f"{ROOT}/quality_by_year")
    exclusions.write.mode("errorifexists").json(f"{ROOT}/exclusion_counts")
    alignment.write.mode("errorifexists").json(f"{ROOT}/alignment_distribution")
    quality.orderBy("year").show(8, truncate=False)
    exclusions.orderBy("_source_year", "exclusion_reason").show(100, truncate=False)
    raw.unpersist()

In [ ]:
with spark_action(spark, RUN_ID, "coverage_and_gaps"):
    clean = spark.read.parquet(f"{ROOT}/cleaned")
    months, years, days, gaps = coverage(clean)
    for name, frame in [("coverage_month", months), ("coverage_year", years), ("coverage_day", days), ("gaps", gaps)]:
        frame.write.mode("errorifexists").parquet(f"{ROOT}/{name}")
        frame.orderBy(*frame.columns[:2]).show(20, truncate=False)
    expected = sum(r["row_count"] for r in manifest["files"])
    excluded = spark.read.json(f"{ROOT}/exclusion_counts").agg(F.sum("count")).first()[0] or 0
    retained = clean.count()
    assert retained + excluded == expected, "Disjoint attrition counts must reconcile to raw rows."
    assert clean.groupBy("station_id", "epoch").count().where("count > 1").limit(1).count() == 0
    display({"raw_rows": expected, "retained_rows": retained, "excluded_rows": excluded})

In [ ]:
spark.stop()
print("Audit and cleaning complete. Review and capture every action's Spark UI evidence.")